In [88]:
import json
import pandas as pd
from collections import defaultdict
from pypdf import PdfReader
import os
import random
import csv
import re
from pathlib import Path


In [89]:
with open("CUAD_v1.json", "r", encoding="utf-8") as f:
    cuad_squad_format = json.load(f)

master_clauses = pd.read_csv("master_clauses.csv")


#### Detailed View

In [90]:
# Show all dataframe columns in outputs
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

In [91]:
master_clauses[master_clauses['Filename'].str.startswith("Conform")]

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,Expiration Date-Answer,Renewal Term,Renewal Term-Answer,Notice Period To Terminate Renewal,Notice Period To Terminate Renewal- Answer,Governing Law,Governing Law-Answer,Most Favored Nation,Most Favored Nation-Answer,Competitive Restriction Exception,Competitive Restriction Exception-Answer,Non-Compete,Non-Compete-Answer,Exclusivity,Exclusivity-Answer,No-Solicit Of Customers,No-Solicit Of Customers-Answer,No-Solicit Of Employees,No-Solicit Of Employees-Answer,Non-Disparagement,Non-Disparagement-Answer,Termination For Convenience,Termination For Convenience-Answer,Rofr/Rofo/Rofn,Rofr/Rofo/Rofn-Answer,Change Of Control,Change Of Control-Answer,Anti-Assignment,Anti-Assignment-Answer,Revenue/Profit Sharing,Revenue/Profit Sharing-Answer,Price Restrictions,Price Restrictions-Answer,Minimum Commitment,Minimum Commitment-Answer,Volume Restriction,Volume Restriction-Answer,Ip Ownership Assignment,Ip Ownership Assignment-Answer,Joint Ip Ownership,Joint Ip Ownership-Answer,License Grant,License Grant-Answer,Non-Transferable License,Non-Transferable License-Answer,Affiliate License-Licensor,Affiliate License-Licensor-Answer,Affiliate License-Licensee,Affiliate License-Licensee-Answer,Unlimited/All-You-Can-Eat-License,Unlimited/All-You-Can-Eat-License-Answer,Irrevocable Or Perpetual License,Irrevocable Or Perpetual License-Answer,Source Code Escrow,Source Code Escrow-Answer,Post-Termination Services,Post-Termination Services-Answer,Audit Rights,Audit Rights-Answer,Uncapped Liability,Uncapped Liability-Answer,Cap On Liability,Cap On Liability-Answer,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
10,ConformisInc_20191101_10-Q_EX-10.6_11861402_EX-10.6_Development Agreement.pdf,['DEVELOPMENT AGREEMENT'],DEVELOPMENT AGREEMENT,"['also known as Stryker Orthopaedics (""Stryker"")', 'Conformis', 'Howmedica Osteonics Corp.', 'Conformis, Inc.', 'Stryker and Conformis are collectively referred to herein as the ""Parties"" and individually as a ""Party.""']","Howmedica Ostenonics Corp. (""Stryker""); Conformis, Inc. (""Conformis""); Stryker and Conformis (“Parties” and individually as a “Party.”)","['September 30, 2019']",9/30/19,"['September 30, 2019']",9/30/19,"['The term of this Agreement shall begin as of the Effective Date and continue until Acceptance of all Deliverables for Milestones #1 and #2 pursuant to Section 3.4 and completion of Milestone #3, unless earlier terminated under Section 8.2, as provided for under the Other Agreements, or as mutually agreed by the Parties.']",NaN,[],NaN,[],NaN,['This Agreement shall be governed and construed in accordance with the laws of New York State (without regard to the conflict of laws provisions thereof).'],New York,[],No,"['For purposes of clarity, the foregoing does not prevent Conformis from granting any license, release, covenant not to sue or other immunity to any third party under any Patents, including any such immunity that would authorize manufacture, use or sale of Patient-Specific Instrumentation for Off-The-Shelf Knee Implants outside the Buyer Field.', 'Except as specifically provided in the Distribution Agreement, Conformis shall be prohibited from developing or assisting another in developing, or causing another to develop, Patient-Specific Instrumentation for Off-The-Shelf Knee Implants for any Third Party in the field of orthopedics until January 1, 2032 (or earlier, to the extent set forth in Section 2.3.3.4 or Section 2.3.5 of the Distribution Agreement), with the exception that Conformis (including any entity involved in a Change of Control of Conformis, any such entity an ""Acquirer""), may develop Patient-Specific Instrumentation for any Off-The- Shelf Implants

In [92]:
el = [qa for qa in cuad_squad_format['data'] if qa['title'].startswith("ConformisInc")][0]

In [93]:
contract_text = el['paragraphs'][0]['context']

In [94]:
contract_text[24600:]

'promptly disclose to Conformis all Joint IP.   (f) To the extent required and for the avoidance of doubt, Stryker hereby grants Conformis, and Conformis hereby   accepts, a non-exclusive license to the Stryker Background IP and Improved Stryker Background IP solely for purposes of   performing any obligations under this Agreement and the Distribution Agreement.   5.2 Confidential Information. The provisions of Sections 4.3(a)-(i) of Article 4 of the APA are incorporated herein as if   fully set forth herein.   5.3 Maintenance of Records. Each Party shall prepare and maintain complete and accurate records concerning all   Inventions for the purpose of documenting any possible Intellectual Property rights arising under this Agreement.   5.4 No Other Rights.   (a) Conformis acknowledges and agrees that, as between the Parties, Stryker owns all right, title and interest, including all   Intellectual Property rights, within the Stryker Background IP and Stryker\'s Confidential Information,

In [95]:
cuad_squad_format['data'][0]['paragraphs'][0]['qas']

[{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'is_impossible': False},
 {'answers': [{'text': 'Distributor', 'answer_start': 244},
   {'text': 'Electric City Corp.', 'answer_start': 148},
   {'text': 'Electric City of Illinois L.L.C.', 'answer_start': 49574},
   {'text': 'Company', 'answer_start': 197},
   {'text': 'Electric City of Illinois LLC', 'answer_start': 212}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties',
  'question': 'Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract',
  'is_impossible': False},
 {'answers': [{'text': '7th day of September, 1999.', 'answer_start': 263}],
  'id': 

In [96]:
[qa for qa in cuad_squad_format['data'][0]['paragraphs'][0]['qas'] if qa['is_impossible'] == True]

[{'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Notice Period To Terminate Renewal',
  'question': 'Highlight the parts (if any) of this contract related to "Notice Period To Terminate Renewal" that should be reviewed by a lawyer. Details: What is the notice period required to terminate renewal?',
  'is_impossible': True},
 {'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Most Favored Nation',
  'question': 'Highlight the parts (if any) of this contract related to "Most Favored Nation" that should be reviewed by a lawyer. Details: Is there a clause that if a third party gets better terms on the licensing or sale of technology/goods/services described in the contract, the buyer of such technology/goods/services under the contract shall be entitled to those better terms?',
  'is_impossible': True},
 {'answers': [],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Non-Compete',
  'question': 'Highlight the parts (if 

In [97]:
cuad_squad_format['data'][0]['paragraphs'][0]['qas']

[{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'is_impossible': False},
 {'answers': [{'text': 'Distributor', 'answer_start': 244},
   {'text': 'Electric City Corp.', 'answer_start': 148},
   {'text': 'Electric City of Illinois L.L.C.', 'answer_start': 49574},
   {'text': 'Company', 'answer_start': 197},
   {'text': 'Electric City of Illinois LLC', 'answer_start': 212}],
  'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties',
  'question': 'Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract',
  'is_impossible': False},
 {'answers': [{'text': '7th day of September, 1999.', 'answer_start': 263}],
  'id': 

#### Distribution View

In [98]:
stats = defaultdict(lambda: {True: 0, False: 0})

# Iteration durch die Datenstruktur
for entry in cuad_squad_format['data']:
    for paragraph in entry.get('paragraphs', []):
        for qa in paragraph.get('qas', []):
            
            # Extraktion des Fragetyps aus der ID (nach dem "__")
            qa_id = qa.get('id', '')
            if "__" in qa_id:
                question_type = qa_id.split("__")[-1]
            else:
                question_type = "Unknown"
            
            # Wert von is_impossible abgreifen (Default False, falls nicht vorhanden)
            is_impossible = qa.get('is_impossible', False)
            
            # Zähler für diesen Typ inkrementieren
            stats[question_type][is_impossible] += 1

# --- Ausgabe der Ergebnisse ---
print(f"{'Fragetyp':<45} | {'True (Nicht vorh.)':<18} | {'False (Vorhanden)':<18}")
print("-" * 85)

# Sortierung der Typen von A-Z für bessere Übersicht
for q_type in sorted(stats.keys()):
    t_count = stats[q_type][True]
    f_count = stats[q_type][False]
    print(f"{q_type:<45} | {t_count:<18} | {f_count:<18}")

Fragetyp                                      | True (Nicht vorh.) | False (Vorhanden) 
-------------------------------------------------------------------------------------
Affiliate License-Licensee                    | 451                | 59                
Affiliate License-Licensor                    | 487                | 23                
Agreement Date                                | 40                 | 470               
Anti-Assignment                               | 136                | 374               
Audit Rights                                  | 296                | 214               
Cap On Liability                              | 235                | 275               
Change Of Control                             | 389                | 121               
Competitive Restriction Exception             | 434                | 76                
Covenant Not To Sue                           | 410                | 100               
Document Name                     

#### Dataset Curation

In [99]:
# KONFIGURATION
WINDOW_SIZE = 400
NOISE_COUNT = 4
SEPARATOR_TOKEN = " [NEXT_SEGMENT] "

QUESTION_TYPE = "License Grant"

OUTPUT_TSV = f"cuad_{QUESTION_TYPE.replace(' ', '_')}.tsv"

In [100]:
def get_question_type_from_id(qa_id: str) -> str:
    """Extrahiert den Fragetyp aus einer QA-ID (Teil nach '__')."""
    return qa_id.split("__")[-1] if "__" in qa_id else "Unknown"


# Nur fuer diese 5 License-Grant-Dokumente soll die ID verkuerzt werden.
# Bitte die Platzhalter-Keys durch die echten Original-Dokument-IDs ersetzen.

LICENSE_GRANT_ID_RENAME_MAP = {}


def extract_qas_by_type(cuad_data, target_types):
    """
    Extrahiert QA-Objekte fuer einen oder mehrere Fragetypen.

    target_types: str oder Iterable[str]
    id_rename_map: optionales Mapping fuer Dokument-IDs (Teil vor '__')
    """
    if isinstance(target_types, str):
        target_types = {target_types}
    else:
        target_types = set(target_types)


    extracted = []
    for entry in cuad_data.get("data", []):
        for paragraph in entry.get("paragraphs", []):
            paragraph_content = paragraph.get("context", "")
            for qa in paragraph.get("qas", []):
                qa_id = qa.get("id", "")
                question_type = get_question_type_from_id(qa_id)

                if question_type in target_types:
                    qa_copy = dict(qa)
                    qa_copy["context"] = paragraph_content

                    if "__" in qa_id:
                        doc_id, _ = qa_id.split("__", 1)
                        
                    extracted.append(qa_copy)
    return extracted


# Wie bisher: nur "License Grant"
extract_qa_list = extract_qas_by_type(
    cuad_squad_format,
    "License Grant",
)

# Optionaler Check
print(f"Gefundene QA-Eintraege: {len(extract_qa_list)}")

Gefundene QA-Eintraege: 510


In [101]:
def merge_segments_by_index(segments):
    """
    Mergt überlappende Segmente basierend auf ihrem Start-Index.
    Eingabe: Liste von Tupeln (start_index, text_content)
    Ausgabe: Liste von nicht-überlappenden Tupeln
    """
    if not segments:
        return []
    
    # Sortiere nach Start-Index
    sorted_segs = sorted(segments, key=lambda x: x[0])
    merged = [sorted_segs[0]]
    
    for start, text in sorted_segs[1:]:
        last_start, last_text = merged[-1]
        # Wenn das neue Segment vor dem Ende des letzten startet, überlappung vorhanden
        if start < last_start + len(last_text):
            # Merge: Nimm den längeren Text oder konkateniere
            merged_text = last_text if len(last_text) >= len(text) else last_text + text[len(last_text):]
            merged[-1] = (last_start, merged_text)
        else:
            merged.append((start, text))
    
    return merged

In [102]:

def get_random_noise(full_text, exclude_ranges, noise_count):
    """
    Extrahiert Random Noise-Segmente aus dem Text.
    Exclude_ranges: Liste von (start, end) Tupeln, die ausgeschlossen werden.
    Gibt Liste von Tupeln (start_index, snippet) zurück.
    """
    noise_snippets = []
    text_length = len(full_text)
    max_attempts = noise_count * 20  # Verhindere Endlosschleife; erhöhe Versuche für robustere Auswahl
    attempts = 0
    
    def overlaps_any(start, end, ranges):
        for r_start, r_end in ranges:
            if not (end <= r_start or start >= r_end):
                return True
        return False
    
    while len(noise_snippets) < noise_count and attempts < max_attempts:
        attempts += 1
        
        # Zufälliger Start-Punkt
        max_start = max(0, text_length - WINDOW_SIZE * 2)
        if max_start <= 0:
            random_start = 0
        else:
            random_start = random.randint(0, max_start)
        
        random_end = min(text_length, random_start + WINDOW_SIZE * 2)
        
        # Prüfe Überschneidung mit bereits ausgeschlossenen Bereichen (gold) sowie bereits gewählten noise-Segmenten
        existing_noise_ranges = [(s, s + len(t)) for s, t in noise_snippets]
        if overlaps_any(random_start, random_end, exclude_ranges) or overlaps_any(random_start, random_end, existing_noise_ranges):
            continue
        
        # Akzeptieren und direkt den Bereich zu exclude_ranges hinzufügen, damit folgende Iterationen ihn vermeiden
        snippet = full_text[random_start:random_end]
        noise_snippets.append((random_start, snippet))
        exclude_ranges.append((random_start, random_end))
    
    return noise_snippets

In [103]:
def save_to_tsv(rows, output_file=OUTPUT_TSV):
    """
    Speichert Daten in einer TSV-Datei.
    rows: Liste von Dictionaries oder Listen mit den Daten
    """
    if not rows:
        return
    
    # Bestimme die Spalten
    if isinstance(rows[0], dict):
        fieldnames = rows[0].keys()
    else:
        fieldnames = ["question", "context"]
    
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

In [104]:

def process_cuad_entry_final(entry, full_text):
    """
    Verarbeitet einen CUAD-Eintrag und erstellt einen Kontext aus:
    1. Gold-Passagen (die tatsächlichen Antworten mit Fenster)
    2. Noise-Passagen (zufällige Segmente außerhalb der Antwort-Bereiche)
    
    Alle Segmente werden nach ihrem Auftreten im Originaltext sortiert
    und mit SEPARATOR_TOKEN verbunden.
    """
    
    # Speichern Tupel: (start_index, text_content)
    all_segments = []
    exclude_ranges = []

    # 1. GOLD-PASSAGEN extrahieren
    if not entry['is_impossible']:
        for ans in entry.get('answers', []):
            answer_start = ans['answer_start']
            answer_text = ans['text']
            
            # Fenster um die Antwort
            s = max(0, answer_start - WINDOW_SIZE)
            e = min(len(full_text), answer_start + len(answer_text) + WINDOW_SIZE)
            
            snippet = full_text[s:e]
            all_segments.append((s, snippet))  # Start-Index für Sortierung merken
            exclude_ranges.append((s, e))
        
        # Optionale Optimierung: Überlappende Gold-Passagen mergen
        if all_segments:
            all_segments = merge_segments_by_index(all_segments)
            # Exclude ranges auch anpassen
            exclude_ranges = [(seg[0], seg[0] + len(seg[1])) for seg in all_segments]

    # 2. RANDOM NOISE extrahieren
    noise_count = NOISE_COUNT if not entry['is_impossible'] else 3
    noise_snippets = get_random_noise(full_text, exclude_ranges, noise_count)
    
    for n_start, n_text in noise_snippets:
        all_segments.append((n_start, n_text))

    # 3. SORTIERUNG (Der entscheidende Schritt)
    # Sortiere alle Segmente (Gold + Noise) nach ihrem Auftreten im Originaldokument
    all_segments.sort(key=lambda x: x[0])

    # 4. KONKATENIEREN MIT TRENNER
    # Nur den Text-Teil der Tupel nehmen
    final_content_list = [seg[1] for seg in all_segments]
    final_text = SEPARATOR_TOKEN.join(final_content_list)

    # 5. CLEANING FÜR TSV
    # Mehrfache Spaces/Tabs auf ein einzelnes Space reduzieren
    normalized_text = re.sub(r"[ \t]{2,}", " ", final_text)
    clean_text = normalized_text.replace("\n", "\\n").replace("\t", "\\t")
    
    return clean_text

In [105]:
# MAIN LOOP
# Extrahiere alle QA-Einträge aus dem CUAD-Dataset

# Nur spezifischen Fragetyp extrahieren (License Grant, Cap on Liability, Audit Rights)


qa_entries = extract_qas_by_type(cuad_squad_format, QUESTION_TYPE)

# Verarbeite alle Einträge
processed_rows = []
failed_entries = []

for i, qa_entry in enumerate(qa_entries):
    qa_id = qa_entry.get('id', '')
    question = qa_entry.get('question', '')
    contract_text = qa_entry.get('context', '')
    
    if not contract_text:
        failed_entries.append(qa_id)
        continue
    
    # Verarbeite den Eintrag
    processed_context = process_cuad_entry_final(qa_entry, contract_text)
    
    # Speichere die verarbeitete Reihe
    processed_rows.append({
        'id': qa_id,
        'question': question,
        'is_impossible': qa_entry.get('is_impossible', False),
        'context': processed_context
    })
    
    if (i + 1) % 100 == 0:
        print(f"Verarbeitet: {i + 1}/{len(qa_entries)}")

print(f"\n✓ Verarbeitung abgeschlossen!")
print(f"Erfolgreiche Einträge: {len(processed_rows)}")
print(f"Fehlende Inhalte: {len(failed_entries)}")

if failed_entries:
    print(f"\nEinträge ohne content-Feld:")
    for qa_id in failed_entries[:10]:
        print(f"  - {qa_id}")
    if len(failed_entries) > 10:
        print(f"  ... und {len(failed_entries) - 10} weitere")

# Speichere in TSV
save_to_tsv(processed_rows)
print(f"\n✓ Ergebnisse gespeichert in: {OUTPUT_TSV}")

Verarbeitet: 100/510
Verarbeitet: 200/510
Verarbeitet: 300/510
Verarbeitet: 400/510
Verarbeitet: 500/510

✓ Verarbeitung abgeschlossen!
Erfolgreiche Einträge: 510
Fehlende Inhalte: 0

✓ Ergebnisse gespeichert in: cuad_License_Grant.tsv


In [80]:
# Folgende Dateien machen Probleme:
# BABCOCK_WILCOXENTERPRISES,INC_08_04_2015-EX-10.17-INTELLECTUAL PROPERTY AGREEMENT between THE BABCOCK _ WILCOX COMPANY and BABCOCK _ WILCOX ENTERPRISES, INC..PDF
# OTISWORLDWIDECORP_04_03_2020-EX-10.4-INTELLECTUAL PROPERTY AGREEMENT by and among UNITED TECHNOLOGIES CORPORATION, OTIS WORLDWIDE CORPORATION and CARRIER ~1.PDF
# WOMENSGOLFUNLIMITEDINC_03_29_2000-EX-10.13-ENDORSEMENT AGREEMENT - Intellectual Property Rights                 Confidentiality and Non-Use Obligations Agreement.PDF
# PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement2.PDF
# PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement1.PDF